In [20]:
import pandas as pd
import numpy as np
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv('train.csv')

In [3]:
numeric_index = []
for i in df.select_dtypes(include=['int', 'float']).columns:
    numeric_index.append(i)

categorical_index = []
for i in df.select_dtypes(include=['bool']).columns:
    categorical_index.append(i)

In [4]:
for i in df.columns:        
    x = df[i].isna().sum()
    if x > 0:
        print(f'{i}: {x}')

LotFrontage: 259
Alley: 1369
MasVnrType: 872
MasVnrArea: 8
BsmtQual: 37
BsmtCond: 37
BsmtExposure: 38
BsmtFinType1: 37
BsmtFinType2: 38
Electrical: 1
FireplaceQu: 690
GarageType: 81
GarageYrBlt: 81
GarageFinish: 81
GarageQual: 81
GarageCond: 81
PoolQC: 1453
Fence: 1179
MiscFeature: 1406


In [5]:
def drop_column(df):
    return df.drop(['YrSold', 'MoSold', 'GarageYrBlt','Utilities','GarageCond','MiscFeature','PoolQC','GarageQual','Exterior1st','Exterior2nd','Heating','BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF','RoofMatl','Condition2','LandSlope','LandContour','Street'], axis=1)


dropper = FunctionTransformer(drop_column)

In [6]:
def fill_column(df):
    col_to_fill = ['PoolQC','BsmtCond','BsmtQual','BsmtExposure','BsmtFinType1','BsmtFinType2',
                   'GarageType','GarageYrBlt','GarageFinish','GarageQual','GarageCond','FireplaceQu',
                   'Alley','Fence','MiscFeature']
    return df['Electrical'].fillna('SBrkr')
    return df[col_to_fill].fillna('NA')
    return df['MasVnrArea'].fillna(0)
    return df['LotFrontage'].fillna(70.05)
    return df['HouseFin'].fillna('Finished')

filler = FunctionTransformer(fill_column)

In [17]:
def make_column(df):
    df = df.to_frame()
    if 'ExteriorMat' in df.columns and 'Exterior2nd' in df.columns:
        df['ExteriorStyle'] = df['ExteriorMat']
        df['ExteriorMat'] = df['Exterior2nd']

    return df

column_maker = FunctionTransformer(make_column)

In [8]:
def label_data(df):
    Label_to_Attchd = ['BuiltIn', 'Basment', '2Types']
    Label_to_Abnorml = ['Alloca','AdjLand','Family']
    Label_to_Fence = ['MnPrv','MnWw','GdWo','GdPrv']
    Label_to_Atyp = ['Min1','Min2','Mod','Maj1','Maj2','Sev']
    Label_to_Fuse = ['FuseA','FuseF','FuseP','Mix']
    Label_to_Uneven = ['Bnk','HLS','Low']
    Label_to_IR = ['IR1','IR2','IR3']
    Label_to_Siding = ['VinylSd','MetalSd','Wd Sdng']
    Label_to_Shingle = ['WdShing','AsbShng','AsphShn']
    SaleType_to_Oth = ['New','COD','ConLD','ConLI','ConLw','Con']
    RoofStyle_to_Oth = ['Flat', 'Gambrel', 'Mansard', 'Shed']
    Foundation_to_Oth = ['Slab', 'Stone', 'Wood']
    ExteriorStyle_to_Other = ['HdBoard','Plywood','CemntBd','Stucco','BrkFace','ImStucc','BrkComm','Stone','CBlock']
    ExteriorMat_to_Other = ['CBlock','AsphShn','Stone','AsbShng','Brick','CemntBd']


    df.loc[df['ExterCond'] == 'TA', ['ExterCond']] = 'Gd'
    df.loc[(df['BsmtFinType2'] == 'NA') & (df['TotalBsmtSF'] > 0), ['BsmtFinType2']] = 'Unf'
    

    for Label in SaleType_to_Oth:
        df.loc[df['SaleType'] == Label, ['SaleType']] = 'Oth'
    
    for label in RoofStyle_to_Oth:
        df.loc[df['RoofStyle'] == label, ['RoofStyle']] = 'Oth'

    for label in Foundation_to_Oth:
        df.loc[df['Foundation'] == label, ['Foundation']] = 'Oth'

    for label in ExteriorStyle_to_Other:
        df.loc[df['ExteriorStyle'] == label,'ExteriorStyle'] = 'Other'

    for label in ExteriorMat_to_Other:
        df.loc[df['ExteriorMat'] == label,'ExteriorStyle'] = 'Other'


    for Label in Label_to_Attchd:
        df.loc[df['GarageType'] == Label, ['GarageType']] = 'Attchd'

    df.loc[df['GarageType'] == 'CarPort', ['GarageType']] = 'Detchd'

    for Label in Label_to_Abnorml:
        df.loc[df['SaleCondition'] == Label, ['SaleCondition']] = 'Abnorml'
    
    for Label in Label_to_Fence:
        df.loc[df['Fence'] == Label, ['Fence']] = 'Fence'

    for label in Label_to_Atyp:
        df.loc[df['Functional'] == label, ['Functional']] = 'Atype'

    for label in Label_to_Fuse:
        df.loc[df['Electrical'] == label, ['Electrical']] = 'Fuse'

    for label in Label_to_Uneven:
        df.loc[df['LandContour'] == label, 'LandContour'] = 'Uneven'

    for label in Label_to_IR:
        df.loc[df['LotShape'] == label,'LotShape'] = 'IR'
    
    for label in Label_to_Siding:
        df.loc[df['ExteriorStyle'] == label,'ExteriorStyle'] = 'Siding'

    for label in Label_to_Shingle:
        df.loc[df['ExteriorStyle'] == label,'ExteriorStyle'] = 'Shingle'
    
    return df

labeller = FunctionTransformer(label_data)

In [18]:
cleaning_pipeline = Pipeline([
                        ('filler', filler),
                        ('column_maker', column_maker),
                        ('labeller', labeller),
                        ('dropper', dropper)
                    ])
cleaning_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('filler', ...), ('column_maker', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function fil...00279A666A0C0>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",None
,"kw_args kw_args: dict, default=NoneDictionary of additional keyword arguments to

In [ ]:
cleaning_pipeline.fit_transform(df)